# Momentum One — SIGON (signals ON · all 4 symbols)

## This is the NEW bot (not the old one)

| | OLD bot | THIS notebook (SIGON) |
|--|---------|------------------------|
| Signal agents | **OFF** | **ON** |
| Brain size | ~1820 | **~6820** |
| Checkpoint | PROVEN_*.pt | **best_sigon*.pt** |
| Load PROVEN into this? | — | **NEVER** |

## Do this in order

1. **Runtime** → **Change runtime type** → **L4** or **T4** → **Save**
2. Run **STEP 1** (play button) — mounts Drive + setup
3. Run **STEP 2** (play button) — **always continues your best Drive brain**
4. Leave it until `upd 1` … (first time building data can take a while)

**STEP 2 will not start a new empty brain.** It copies from Drive first.

Open again later:  
https://colab.research.google.com/github/monty313/the-truth/blob/main/GPU_EDITION/Momentum_One_RunAll.ipynb


# STEP 1 — Setup (run once)

- Connects Google Drive
- Downloads code
- Copies **XAUUSD + EURUSD + GBPUSD + US30**
- Checks that **signal agents are ON** (stops if old bot config)

Click play. When asked, click **Allow** for Drive.


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 or T4 > Save')

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, re

!git clone https://github.com/monty313/the-truth.git 2>/dev/null || true
%cd /content/the-truth
!git pull origin main
!pip -q install pyyaml >/dev/null 2>&1

# HARD CHECK: signals ON (SIGON). Refuse old bot config.
feat_path = 'configs/features.yaml'
feat_txt = open(feat_path, encoding='utf-8').read()
m = re.search(r'^include_signal_agent_slots:\s*(\w+)', feat_txt, re.M)
sig_on = (m and m.group(1).lower() == 'true')
print()
print('=== LINEAGE CHECK ===')
print('include_signal_agent_slots:', m.group(1) if m else 'MISSING')
if not sig_on:
    raise SystemExit('STOP: signal agents are OFF — that is the OLD bot (~1820). Need true in configs/features.yaml.')
print('OK — SIGON path (signals ON, new ~6820 brain). Will NOT load PROVEN 1820.')
print('=====================')
print()

candidates = [
    '/content/drive/MyDrive/Camillion_data',
    '/content/drive/MyDrive/the-truth-data',
    '/content/drive/MyDrive/MOMENTUM_ONE/02_PRICE_DATA',
]
src = None
for c in candidates:
    if os.path.isdir(c) and glob.glob(c + '/**/*.csv', recursive=True):
        src = c
        break
if src is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        names = ' '.join(files).upper()
        if 'XAUUSD' in names and any(x in names for x in ('EURUSD', 'US30', 'GBPUSD')):
            if any(f.lower().endswith('.csv') for f in files):
                src = root
                break

os.makedirs('data', exist_ok=True)
copied = []
if src:
    print('Using price folder:', src)
    for f in glob.glob(src + '/**/*.csv', recursive=True):
        base = os.path.basename(f).upper()
        if any(s in base for s in ('XAUUSD', 'EURUSD', 'GBPUSD', 'US30')):
            dst = os.path.join('data', os.path.basename(f))
            shutil.copy2(f, dst)
            copied.append(os.path.basename(dst))
            print('  copied', os.path.basename(dst))
else:
    print('WARNING: no price CSVs found on Drive.')
    print('Put XAUUSD / EURUSD / GBPUSD / US30 CSVs in Drive folder Camillion_data')

print()
print('Files in data/:')
for name in sorted(os.listdir('data')):
    if name.lower().endswith('.csv'):
        mb = os.path.getsize(os.path.join('data', name)) / (1024 * 1024)
        print(' ', name, '  %.0f MB' % mb)

need = ['XAUUSD', 'EURUSD', 'GBPUSD', 'US30']
have = ' '.join(copied).upper() if copied else ' '.join(os.listdir('data')).upper()
missing = [s for s in need if s not in have]
if missing:
    print('MISSING symbols:', missing)
else:
    print('All 4 symbols present. Good.')

# Clear feature caches only — never deletes PROVEN or any .pt brains
!rm -f artifacts/gpu_cache_*.npz
!rm -rf artifacts/symbol_cache
print()
print('Setup done. Now run STEP 2 (all 4 symbols, SIGON).')


# STEP 2 — Train (continue best brain only)

Press **▶** once. Leave it running.

### What this does (automatic)
1. Checks **signals ON**
2. Copies your best brain from **Google Drive** into Colab
3. Starts training **only if** that brain loaded (no new empty brain)
4. Uses **4000** parallel games (bigger practice)

### Good lines to see
- `restored from Drive: best_sigon.pt` or `Drive champion ready`
- **`warm-start best_sigon (obs_dim=6820)`**
- then `upd 1` `upd 2` ...

### Bad line (cell will STOP)
- `STOP: --require-warm` or `STOP: no champion`

### If red out-of-memory (OOM)
In the last line of the code cell, change `4000` to `2000` or `1000`, press ▶ again.


In [ ]:
%cd /content/the-truth
!git pull origin main

import re, os, shutil, glob

# --- 1) Signals must be ON ---
t = open('configs/features.yaml', encoding='utf-8').read()
m = re.search(r'^include_signal_agent_slots:\s*(\w+)', t, re.M)
print('include_signal_agent_slots =', m.group(1) if m else 'MISSING')
if not m or m.group(1).lower() != 'true':
    raise SystemExit('STOP: signals OFF — old bot. Do not train.')

# --- 2) Always copy best brain from Drive (your saved streak) ---
ck = 'artifacts/checkpoints'
os.makedirs(ck, exist_ok=True)
drive = '/content/drive/MyDrive/momentum_sigon_champs'
if not os.path.isdir(drive):
    raise SystemExit(
        'STOP: Drive folder missing: ' + drive + '\n'
        '  Run STEP 1 (mount Drive) first. Or run STEP 4 after a good train to create it.'
    )
found = glob.glob(drive + '/best_sigon*.pt')
if not found:
    raise SystemExit(
        'STOP: no best_sigon files in Drive.\n'
        '  Expected: /content/drive/MyDrive/momentum_sigon_champs/best_sigon.pt'
    )
for fp in found:
    shutil.copy2(fp, os.path.join(ck, os.path.basename(fp)))
    print('restored from Drive:', os.path.basename(fp))
local = os.path.join(ck, 'best_sigon.pt')
if not os.path.isfile(local):
    raise SystemExit('STOP: best_sigon.pt still missing after Drive restore.')
print('Drive champion ready:', local)

# --- 3) Train from that brain only (refuse NEW empty brain) ---
# instances=4000  |  if OOM red error: change 4000 -> 2000 or 1000
!python scripts/gpu_train.py --csv-dir data --symbols XAUUSD,EURUSD,GBPUSD,US30 --instances 4000 --env-mb 32 --max-days-per-symbol 120 --minutes 600 --entropy-coef 0.03 --warm best_sigon --require-warm


# STEP 3 — Status (optional)

Only after you **stop** STEP 2, or after training ends.

Then re-run STEP 2 to continue.


In [ ]:
%cd /content/the-truth
!python scripts/jarvis_talk.py status
!python scripts/jarvis_talk.py board
print('--- progress ---')
!cat artifacts/checkpoints/gpu_progress.json 2>/dev/null || echo 'no progress yet'
print('--- SIGON champions (not PROVEN) ---')
!ls -1 artifacts/checkpoints/best_sigon*.pt 2>/dev/null || echo 'no champion yet'


# STEP 4 — Copy champion to Drive (optional)


In [ ]:
import os, shutil, glob
%cd /content/the-truth
dst = '/content/drive/MyDrive/momentum_sigon_champs'
os.makedirs(dst, exist_ok=True)
found = glob.glob('artifacts/checkpoints/best_sigon*.pt')
if not found:
    print('No best_sigon files yet. Keep training in STEP 2.')
else:
    for p in found:
        shutil.copy2(p, os.path.join(dst, os.path.basename(p)))
        print('saved', os.path.basename(p))
    print('Drive folder:', dst)
